# Model Ensemble
### Model_Comparison에서 성능이 좋았던 CatBoost와 LightGBM을 앙상블

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier

In [2]:
# =========================
# 데이터 지정 (top30)
# =========================
DATA_DIR = Path('../Data')
train_df = pd.read_csv(DATA_DIR / "train_selection_top30.csv")
test_df = pd.read_csv(DATA_DIR / "test_selection_top30.csv")

target_col = "fraud"
# =========================
# Train / Test 분리
# =========================

X = train_df.drop(columns=[target_col]).copy()
y = train_df[target_col].copy()

X_test = test_df.copy()   # test에는 target 없음

print("Train X shape:", X.shape)
print("Train y shape:", y.shape)
print("Test X shape :", X_test.shape)
print("Positive ratio:", y.mean().round(4))

Train X shape: (18000, 30)
Train y shape: (18000,)
Test X shape : (12000, 30)
Positive ratio: 0.1582


In [3]:
# 평가 함수
def evaluate_scores(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0)
    }


def recall_at_k(y_true, y_prob, top_ratio=0.10):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    n_top = int(len(y_prob) * top_ratio)
    n_top = max(n_top, 1)

    top_idx = np.argsort(y_prob)[::-1][:n_top]
    total_positive = y_true.sum()

    if total_positive == 0:
        return 0.0

    return y_true[top_idx].sum() / total_positive

In [4]:
# LightGBM + CatBoost OOF 학습 함수
def run_lgb_cat_oof(X, y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    lgb_oof = np.zeros(len(X))
    cat_oof = np.zeros(len(X))

    lgb_fold_scores = []
    cat_fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n{'='*20} Fold {fold} {'='*20}")

        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_valid = y.iloc[valid_idx].copy()

        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()
        scale_pos_weight = (neg / pos) * 1.1 if pos > 0 else 1.0

        # -------------------------
        # LightGBM
        # -------------------------
        lgb_model = LGBMClassifier(
            objective="binary",
            metric="auc",
            boosting_type="gbdt",
            learning_rate=0.03,
            num_leaves=64,
            max_depth=-1,
            min_child_samples=20,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.5,
            n_estimators=10000,
            random_state=random_state,
            n_jobs=-1,
            scale_pos_weight=scale_pos_weight,
            verbosity=-1,
        )

        lgb_model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            callbacks=[
                early_stopping(stopping_rounds=300, verbose=False),
                log_evaluation(period=0)
            ]
        )

        lgb_valid_prob = lgb_model.predict_proba(X_valid)[:, 1]
        lgb_oof[valid_idx] = lgb_valid_prob

        lgb_fold_result = {
            "fold": fold,
            "roc_auc": roc_auc_score(y_valid, lgb_valid_prob),
            "pr_auc": average_precision_score(y_valid, lgb_valid_prob)
        }
        lgb_fold_scores.append(lgb_fold_result)

        # -------------------------
        # CatBoost
        # -------------------------
        cat_model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            learning_rate=0.03,
            iterations=5000,
            depth=4,
            l2_leaf_reg=5,
            bootstrap_type="Bernoulli",
            subsample=1.0,
            scale_pos_weight=scale_pos_weight,
            random_seed=random_state,
            verbose=-1
        )

        cat_model.fit(
            X_train, 
            y_train,
            eval_set=(X_valid, y_valid),
            verbose=False
        )

        cat_valid_prob = cat_model.predict_proba(X_valid)[:, 1]
        cat_oof[valid_idx] = cat_valid_prob

        cat_fold_result = {
            "fold": fold,
            "roc_auc": roc_auc_score(y_valid, cat_valid_prob),
            "pr_auc": average_precision_score(y_valid, cat_valid_prob)
        }
        cat_fold_scores.append(cat_fold_result)

        print(f"[LGB] ROC-AUC: {lgb_fold_result['roc_auc']:.4f} | PR-AUC: {lgb_fold_result['pr_auc']:.4f}")
        print(f"[CAT] ROC-AUC: {cat_fold_result['roc_auc']:.4f} | PR-AUC: {cat_fold_result['pr_auc']:.4f}")

    return lgb_oof, cat_oof, pd.DataFrame(lgb_fold_scores), pd.DataFrame(cat_fold_scores)

In [5]:
# 학습 실행
lgb_oof, cat_oof, lgb_fold_df, cat_fold_df = run_lgb_cat_oof(
    X=X,
    y=y,
    n_splits=5,
    random_state=2542
)


==================== Fold 1 ====================
[LGB] ROC-AUC: 0.6860 | PR-AUC: 0.2868
[CAT] ROC-AUC: 0.6900 | PR-AUC: 0.2905

==================== Fold 2 ====================
[LGB] ROC-AUC: 0.7042 | PR-AUC: 0.3076
[CAT] ROC-AUC: 0.7304 | PR-AUC: 0.3381

==================== Fold 3 ====================
[LGB] ROC-AUC: 0.6968 | PR-AUC: 0.2832
[CAT] ROC-AUC: 0.7061 | PR-AUC: 0.2875

==================== Fold 4 ====================
[LGB] ROC-AUC: 0.6947 | PR-AUC: 0.2916
[CAT] ROC-AUC: 0.7084 | PR-AUC: 0.3006

==================== Fold 5 ====================
[LGB] ROC-AUC: 0.7031 | PR-AUC: 0.2868
[CAT] ROC-AUC: 0.7108 | PR-AUC: 0.3006


In [6]:
# 개별 모델 성능 확인
lgb_result_05 = evaluate_scores(y, lgb_oof, threshold=0.5)
cat_result_05 = evaluate_scores(y, cat_oof, threshold=0.5)

print("=== LightGBM OOF ===")
print(lgb_result_05)

print("\n=== CatBoost OOF ===")
print(cat_result_05)

=== LightGBM OOF ===
{'roc_auc': 0.695389389393472, 'pr_auc': 0.2845199779363633, 'f1': 0.36236771717641586, 'precision': 0.2709094070596418, 'recall': 0.5470505617977528}

=== CatBoost OOF ===
{'roc_auc': 0.708883547468054, 'pr_auc': 0.2995215301107077, 'f1': 0.36146707909854175, 'precision': 0.24152592417621355, 'recall': 0.7180477528089888}


In [7]:
# 단순 평균 앙상블
ensemble_avg_oof = 0.5 * lgb_oof + 0.5 * cat_oof

ens_avg_result_05 = evaluate_scores(y, ensemble_avg_oof, threshold=0.5)

print("=== Ensemble (0.5 / 0.5) ===")
print(ens_avg_result_05)

=== Ensemble (0.5 / 0.5) ===
{'roc_auc': 0.7053745825077418, 'pr_auc': 0.2951092959788404, 'f1': 0.366073255576018, 'precision': 0.25830205024545194, 'recall': 0.6281601123595506}


In [8]:
# threshold 0.3 기준
lgb_result_03 = evaluate_scores(y, lgb_oof, threshold=0.3)
cat_result_03 = evaluate_scores(y, cat_oof, threshold=0.3)
ens_avg_result_03 = evaluate_scores(y, ensemble_avg_oof, threshold=0.3)

print("=== Threshold = 0.3 ===")
print("LightGBM :", lgb_result_03)
print("CatBoost :", cat_result_03)
print("Ensemble :", ens_avg_result_03)

=== Threshold = 0.3 ===
LightGBM : {'roc_auc': 0.695389389393472, 'pr_auc': 0.2845199779363633, 'f1': 0.3303434987708458, 'precision': 0.20372039662378102, 'recall': 0.8728932584269663}
CatBoost : {'roc_auc': 0.708883547468054, 'pr_auc': 0.2995215301107077, 'f1': 0.30527360988526037, 'precision': 0.18108638743455496, 'recall': 0.9715589887640449}
Ensemble : {'roc_auc': 0.7053745825077418, 'pr_auc': 0.2951092959788404, 'f1': 0.31968295904887717, 'precision': 0.19281471823844706, 'recall': 0.9346910112359551}


In [9]:
# Top-K 비교
topk_df = pd.DataFrame({
    "model": ["LightGBM", "CatBoost", "Ensemble_0.5_0.5"],
    "Recall@Top5%": [
        recall_at_k(y, lgb_oof, 0.05),
        recall_at_k(y, cat_oof, 0.05),
        recall_at_k(y, ensemble_avg_oof, 0.05)
    ],
    "Recall@Top10%": [
        recall_at_k(y, lgb_oof, 0.10),
        recall_at_k(y, cat_oof, 0.10),
        recall_at_k(y, ensemble_avg_oof, 0.10)
    ],
    "Recall@Top20%": [
        recall_at_k(y, lgb_oof, 0.20),
        recall_at_k(y, cat_oof, 0.20),
        recall_at_k(y, ensemble_avg_oof, 0.20)
    ]
})

print(topk_df.round(4))

              model  Recall@Top5%  Recall@Top10%  Recall@Top20%
0          LightGBM        0.1180         0.2166         0.3862
1          CatBoost        0.1310         0.2321         0.3961
2  Ensemble_0.5_0.5        0.1299         0.2303         0.3919


In [10]:
# PR-AUC 기준 최적 weight 탐색
weight_results = []

for w in np.arange(0.0, 1.01, 0.05):
    ens_oof = w * lgb_oof + (1 - w) * cat_oof

    row = {
        "lgb_weight": round(w, 2),
        "cat_weight": round(1 - w, 2),
        "roc_auc": roc_auc_score(y, ens_oof),
        "pr_auc": average_precision_score(y, ens_oof),
        "recall_top5": recall_at_k(y, ens_oof, 0.05),
        "recall_top10": recall_at_k(y, ens_oof, 0.10),
        "recall_top20": recall_at_k(y, ens_oof, 0.20),
    }
    weight_results.append(row)

weight_df = pd.DataFrame(weight_results).sort_values(
    by=["pr_auc", "roc_auc"], ascending=False
).reset_index(drop=True)

print(weight_df.round(4))

    lgb_weight  cat_weight  roc_auc  pr_auc  recall_top5  recall_top10  \
0         0.00        1.00   0.7089  0.2995       0.1310        0.2321   
1         0.05        0.95   0.7090  0.2994       0.1320        0.2314   
2         0.10        0.90   0.7089  0.2992       0.1306        0.2310   
3         0.15        0.85   0.7088  0.2992       0.1306        0.2317   
4         0.20        0.80   0.7086  0.2989       0.1320        0.2310   
5         0.25        0.75   0.7082  0.2985       0.1324        0.2317   
6         0.30        0.70   0.7078  0.2980       0.1324        0.2321   
7         0.35        0.65   0.7073  0.2975       0.1324        0.2317   
8         0.40        0.60   0.7068  0.2967       0.1320        0.2324   
9         0.45        0.55   0.7061  0.2960       0.1313        0.2310   
10        0.50        0.50   0.7054  0.2951       0.1299        0.2303   
11        0.55        0.45   0.7046  0.2943       0.1296        0.2296   
12        0.60        0.40   0.7038  0

In [11]:
# best weight로 최종 앙상블
best_row = weight_df.iloc[0]
best_lgb_w = best_row["lgb_weight"]
best_cat_w = best_row["cat_weight"]

best_ensemble_oof = best_lgb_w * lgb_oof + best_cat_w * cat_oof

print(f"Best weight -> LightGBM: {best_lgb_w}, CatBoost: {best_cat_w}")

Best weight -> LightGBM: 0.0, CatBoost: 1.0


In [12]:
# best ensemble 최종 평가
best_result_05 = evaluate_scores(y, best_ensemble_oof, threshold=0.5)
best_result_03 = evaluate_scores(y, best_ensemble_oof, threshold=0.3)

print("=== Best Ensemble (threshold=0.5) ===")
print(best_result_05)

print("\n=== Best Ensemble (threshold=0.3) ===")
print(best_result_03)

print("\n=== Best Ensemble Top-K ===")
print("Recall@Top5% :", round(recall_at_k(y, best_ensemble_oof, 0.05), 4))
print("Recall@Top10%:", round(recall_at_k(y, best_ensemble_oof, 0.10), 4))
print("Recall@Top20%:", round(recall_at_k(y, best_ensemble_oof, 0.20), 4))

=== Best Ensemble (threshold=0.5) ===
{'roc_auc': 0.708883547468054, 'pr_auc': 0.2995215301107077, 'f1': 0.36146707909854175, 'precision': 0.24152592417621355, 'recall': 0.7180477528089888}

=== Best Ensemble (threshold=0.3) ===
{'roc_auc': 0.708883547468054, 'pr_auc': 0.2995215301107077, 'f1': 0.30527360988526037, 'precision': 0.18108638743455496, 'recall': 0.9715589887640449}

=== Best Ensemble Top-K ===
Recall@Top5% : 0.131
Recall@Top10%: 0.2321
Recall@Top20%: 0.3961


In [13]:
# 최종 비교표
summary_df = pd.DataFrame([
    {
        "model": "LightGBM",
        "roc_auc": roc_auc_score(y, lgb_oof),
        "pr_auc": average_precision_score(y, lgb_oof),
        "recall_top10": recall_at_k(y, lgb_oof, 0.10)
    },
    {
        "model": "CatBoost",
        "roc_auc": roc_auc_score(y, cat_oof),
        "pr_auc": average_precision_score(y, cat_oof),
        "recall_top10": recall_at_k(y, cat_oof, 0.10)
    },
    {
        "model": "Ensemble_0.5_0.5",
        "roc_auc": roc_auc_score(y, ensemble_avg_oof),
        "pr_auc": average_precision_score(y, ensemble_avg_oof),
        "recall_top10": recall_at_k(y, ensemble_avg_oof, 0.10)
    },
    {
        "model": f"Best_Ensemble_{best_lgb_w:.2f}_{best_cat_w:.2f}",
        "roc_auc": roc_auc_score(y, best_ensemble_oof),
        "pr_auc": average_precision_score(y, best_ensemble_oof),
        "recall_top10": recall_at_k(y, best_ensemble_oof, 0.10)
    }
])

print(summary_df.round(4).sort_values(by=["pr_auc", "roc_auc"], ascending=False))

                     model  roc_auc  pr_auc  recall_top10
1                 CatBoost   0.7089  0.2995        0.2321
3  Best_Ensemble_0.00_1.00   0.7089  0.2995        0.2321
2         Ensemble_0.5_0.5   0.7054  0.2951        0.2303
0                 LightGBM   0.6954  0.2845        0.2166


#### CatBoost가 LightGBM을 압도하기 때문에 앙상블 모델 대신 CatBoost 단일 모델을 최종 모델로 확정